# Soft-link toy: PEF ↔ FACE via foro/comarca + data (±N dias)

**Resumo.** Após a escada de match por chave dura (`numero` / `num_processo_limpo`) mostrar cobertura baixa (~2–4%), este notebook testa um **soft-link determinístico por bloqueio**: comarca/foro normalizado ≈ `NOMECOMARCA`↔`foro` e `DT_AJUIZAMENTO` ≈ `distribuicao_data` ± $N$ dias, só sobre o subconjunto **unmatched**. Não implementamos EM nem pesos $m$/$u$ de Fellegi–Sunter (1969) — usamos o framing clássico só para situar *blocking* vs *exact match*.

**Escopo (baixo custo).** Janela intermediária 2017–2020; amostra estratificada ~300 PEFs; DuckDB `IN` ≤300 para hard match; soft-link só nos unmatched (cap ~150). Sem PII (sem nomes/réus em tabelas). Sem commit; sem writes em habitual-tax.

**Figuras.** Plotly interativo (`fig.show`) + PNG estático via Kaleido em `.local/figures/litigancia/` (playground = só `.ipynb`).


## 1. Definição formal do soft-link (bloqueio determinístico)

Seja $\mathcal{U}$ o conjunto de PEFs **sem** hit em FACE por chave dura
$K \in \{\texttt{numero},\ \texttt{num_processo_limpo}\}$ (digits-only / máscara).

Para cada $i\in\mathcal{U}$ e janela $N\in\{0,3,7,15,30\}$ dias, o **bloco** é

$$
B_N(i)=\bigl\{j\in\mathrm{FACE}:\
\phi(\mathrm{NOMECOMARCA}_i)=\phi(\mathrm{foro}_j)
\ \wedge\
\bigl|\mathrm{DT\_AJUIZAMENTO}_i - \mathrm{distribuicao\_data}_j\bigr| \le N
\bigr\},
$$

onde $\phi$ normaliza strings (maiúsculas, sem acento, remove prefixos `FORO DE/DAS`, `COMARCA DE`, mapeia Capital estadual).

**Classificação (1:1 vs ambíguo):**

$$
\begin{aligned}
\mathrm{unique}_N(i) &= \mathbf{1}\{|B_N(i)|=1\},\\
\mathrm{ambiguous}_N(i) &= \mathbf{1}\{|B_N(i)|>1\},\\
\mathrm{none}_N(i) &= \mathbf{1}\{|B_N(i)|=0\}.
\end{aligned}
$$

Taxas sobre $U=|\mathcal{U}|$: $\hat{u}_N=\frac{1}{U}\sum_i\mathrm{unique}_N(i)$,
$\hat{a}_N=\frac{1}{U}\sum_i\mathrm{ambiguous}_N(i)$.
Lift vs hard-only: cobertura enriquecida se unique soft-links fossem aceitos,
$\hat{c}_{\mathrm{hard+u}}=(M_{\mathrm{hard}}+U\cdot\hat{u}_N)/N_{\mathrm{join}}$.

**Cautela (record linkage).** Fellegi & Sunter (1969) e Christen (2012) alertam que bloqueio frouxo eleva falsos positivos; aqui **não** estimamos pesos probabilísticos — reportamos tamanhos de bloco e ambiguidade como diagnóstico de ruído. Isto é **blocking determinístico**, não linkage probabilístico.


## 2. Setup


In [1]:
from __future__ import annotations

import json
import os
import re
import sys
import unicodedata
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype"
from dotenv import load_dotenv

NB_DIR = Path.cwd()
MONO = next((p for p in [NB_DIR, *NB_DIR.parents] if (p / "shared" / "cemepi_api").exists()), NB_DIR)
sys.path.insert(0, str(MONO))
sys.path.insert(0, str(MONO / "pipelines" / "litigancia" / "src"))
sys.path.insert(0, str(MONO / "shared"))
os.chdir(MONO)
load_dotenv(MONO / ".env")

from config.paths import REPO_ROOT, SILVER_FACE_CLEAN, LAKE_ROOT
from cemepi_api.client import CemepiClient, load_settings

AJUIZ_OVERSAMPLE = 2000
PEF_JOIN_CAP = 300  # DuckDB IN ≤300
SOFT_UNMATCHED_CAP = 150
DUCKDB_MEM = "2GB"
RANDOM_STATE = 42
WINDOW_START = pd.Timestamp("2017-01-01")
WINDOW_END = pd.Timestamp("2020-12-31")
N_DAYS = [0, 3, 7, 15, 30]
MAX_N = max(N_DAYS)

assert REPO_ROOT == MONO
assert SILVER_FACE_CLEAN.exists(), f"missing FACE silver: {SILVER_FACE_CLEAN}"

S = load_settings()
C = CemepiClient(S)
SAMPLE_DIR = C.sample_dir()
DUMP_ROOT = S.dump_root

FIG_DIR = MONO / ".local" / "figures" / "litigancia"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def show_and_save(fig, stem: str, *, width: int = 960, height: int | None = None) -> Path:
    # Interactive show + static PNG via Kaleido (playground = ipynb only).
    if height is not None:
        fig.update_layout(height=height)
    fig.update_layout(width=width, template="plotly_white", font=dict(size=12))
    fig.show()
    png = FIG_DIR / f"{stem}.png"
    kw = dict(width=width, scale=2)
    if height is not None:
        kw["height"] = height
    fig.write_image(str(png), **kw)
    print(f"static → {png}")
    return png


print("REPO_ROOT", REPO_ROOT)
print("extracao", f"{S.ano:04d}-{S.mes:02d}")
print("SILVER_FACE_CLEAN", SILVER_FACE_CLEAN.name)
print("FIG_DIR", FIG_DIR)
print("window", WINDOW_START.date(), "…", WINDOW_END.date())
print("caps", {"pef_join": PEF_JOIN_CAP, "soft_unmatched": SOFT_UNMATCHED_CAP, "N_days": N_DAYS})


REPO_ROOT /Users/etorebraga/Code/cemepi-ctf-intel-fiscal
extracao 2026-03
SILVER_FACE_CLEAN face_processos_clean_delta
FIG_DIR /Users/etorebraga/Code/cemepi-ctf-intel-fiscal/.local/figures/litigancia
window 2017-01-01 … 2020-12-31
caps {'pef_join': 300, 'soft_unmatched': 150, 'N_days': [0, 3, 7, 15, 30]}


## 3. Carga — ajuizamento mid-window + amostra estratificada

Preferimos o parquet já materializado pelo coverage toy; se ausente, reconstituímos via API/dump samples. Sem imprimir nomes de partes.


In [2]:
def _normalize_aj(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    need = ["ID_DEBITO", "PEF", "DT_AJUIZAMENTO", "NOMECOMARCA"]
    for c in need:
        if c not in out.columns:
            out[c] = pd.NA
    out = out[need]
    out["PEF"] = out["PEF"].astype(str).str.strip().replace({"nan": pd.NA, "None": pd.NA})
    out["DT_AJUIZAMENTO"] = pd.to_datetime(out["DT_AJUIZAMENTO"], errors="coerce")
    return out


mid_path = SAMPLE_DIR / "toy_ajuizamento_midwindow.parquet"
if mid_path.exists():
    aj = _normalize_aj(pd.read_parquet(mid_path))
    print("loaded mid-window parquet", mid_path.name, "rows", len(aj))
else:
    frames: list[pd.DataFrame] = []
    try:
        frames.append(
            _normalize_aj(
                C.sample_dataset(
                    "ajuizamento",
                    limit=AJUIZ_OVERSAMPLE,
                    select="ID_DEBITO,PEF,DT_AJUIZAMENTO,NOMECOMARCA",
                )
            ).assign(_src="api")
        )
    except Exception as e:  # noqa: BLE001
        print("API failed:", repr(e))
    local_dir = DUMP_ROOT / f"extracao={S.ano:04d}-{S.mes:02d}" / "samples"
    for p in sorted(local_dir.glob("*ajuizamento*.parquet")) if local_dir.exists() else []:
        try:
            frames.append(_normalize_aj(pd.read_parquet(p)).assign(_src=f"dump:{p.name}"))
        except Exception as e:  # noqa: BLE001
            print("skip", p.name, e)
    assert frames, "no ajuizamento source"
    aj = pd.concat(frames, ignore_index=True).dropna(subset=["PEF"]).drop_duplicates("PEF")
    aj = aj.loc[aj["DT_AJUIZAMENTO"].between(WINDOW_START, WINDOW_END, inclusive="both")].copy()
    mid_path.parent.mkdir(parents=True, exist_ok=True)
    aj.to_parquet(mid_path, index=False)
    print("built mid-window", len(aj), "→", mid_path.name)

aj = aj.dropna(subset=["PEF"]).copy()
aj["PEF"] = aj["PEF"].astype(str).str.strip()
aj = aj.loc[aj["PEF"].str.len() > 0].drop_duplicates("PEF", keep="first")
aj = aj.loc[aj["DT_AJUIZAMENTO"].between(WINDOW_START, WINDOW_END, inclusive="both")].copy()
aj["year"] = aj["DT_AJUIZAMENTO"].dt.year

print("mid-window unique PEF", len(aj))
print(aj["year"].value_counts().sort_index().to_string())

vc_year = aj["year"].value_counts().sort_index()
alloc = ((vc_year / vc_year.sum()) * PEF_JOIN_CAP).round().astype(int)
while int(alloc.sum()) > PEF_JOIN_CAP:
    alloc.iloc[int(alloc.argmax())] -= 1
while int(alloc.sum()) < PEF_JOIN_CAP and int(alloc.sum()) < len(aj):
    rem = vc_year - alloc
    if int(rem.max()) <= 0:
        break
    alloc.iloc[int(rem.argmax())] += 1

parts = []
for y, n in alloc.items():
    pool = aj.loc[aj["year"] == y]
    parts.append(pool.sample(n=min(int(n), len(pool)), random_state=RANDOM_STATE))
aj_join = pd.concat(parts, ignore_index=True)
print("join-set n", len(aj_join), "alloc", alloc.to_dict())


loaded mid-window parquet toy_ajuizamento_midwindow.parquet rows 768
mid-window unique PEF 768
year
2017    143
2018    409
2019     73
2020    143
join-set n 300 alloc {2017: 56, 2018: 159, 2019: 29, 2020: 56}


## 4. Hard match (digits-only / `numero`) → matched vs unmatched

Mesma lógica do coverage toy: DuckDB `WHERE numero IN (…) OR digits IN (…)`. Soft-link **só** no unmatched.


In [3]:
def digits_only(s: str) -> str:
    return re.sub(r"\D", "", s or "")


pef_df = pd.DataFrame({"pef": aj_join["PEF"].tolist()})
pef_df["pef_digits"] = pef_df["pef"].map(digits_only)
n_pef = len(pef_df)
assert n_pef <= PEF_JOIN_CAP

con = duckdb.connect()
con.execute(f"SET memory_limit='{DUCKDB_MEM}'")
con.register("pef_list", pef_df[["pef", "pef_digits"]])
face_path = str(SILVER_FACE_CLEAN).replace("'", "''")

face_hit = con.execute(
    "SELECT DISTINCT "
    "CAST(numero AS VARCHAR) AS numero, "
    "regexp_replace(CAST(numero AS VARCHAR), '[^0-9]', '', 'g') AS digits, "
    "CAST(foro AS VARCHAR) AS foro, "
    "distribuicao_data "
    f"FROM delta_scan('{face_path}') "
    "WHERE CAST(numero AS VARCHAR) IN (SELECT pef FROM pef_list) "
    "OR regexp_replace(CAST(numero AS VARCHAR), '[^0-9]', '', 'g') IN (SELECT pef_digits FROM pef_list)"
).fetchdf()

face_by_num = {str(r["numero"]): r for r in face_hit.to_dict("records")}
face_by_dig: dict[str, dict] = {}
for r in face_hit.to_dict("records"):
    face_by_dig.setdefault(str(r["digits"]), r)

rows = []
for r in pef_df.itertuples():
    hit = face_by_num.get(r.pef) or face_by_dig.get(r.pef_digits)
    rows.append({"pef": r.pef, "hard_matched": hit is not None})
hard = pd.DataFrame(rows)
n_hard = int(hard["hard_matched"].sum())
rate_hard = n_hard / n_pef if n_pef else float("nan")
print(f"hard match (exact∪digits): {n_hard}/{n_pef} = {rate_hard:.1%}")
print("FACE hard-hit rows:", len(face_hit))

aj_cov = aj_join.merge(hard, left_on="PEF", right_on="pef", how="left")
aj_cov["hard_matched"] = aj_cov["hard_matched"].fillna(False).astype(bool)

unm = aj_cov.loc[~aj_cov["hard_matched"]].copy()
if len(unm) > SOFT_UNMATCHED_CAP:
    vu = unm["year"].value_counts().sort_index()
    al = ((vu / vu.sum()) * SOFT_UNMATCHED_CAP).round().astype(int)
    while int(al.sum()) > SOFT_UNMATCHED_CAP:
        al.iloc[int(al.argmax())] -= 1
    while int(al.sum()) < SOFT_UNMATCHED_CAP and int(al.sum()) < len(unm):
        rem = vu - al
        if int(rem.max()) <= 0:
            break
        al.iloc[int(rem.argmax())] += 1
    parts_u = []
    for y, n in al.items():
        pool = unm.loc[unm["year"] == y]
        parts_u.append(pool.sample(n=min(int(n), len(pool)), random_state=RANDOM_STATE))
    unm = pd.concat(parts_u, ignore_index=True)
    print("soft unmatched capped alloc", al.to_dict())

U = len(unm)
print(f"unmatched soft-link candidates U={U} (of {int((~aj_cov['hard_matched']).sum())} unmatched in join set)")
print("unmatched year counts:\n", unm["year"].value_counts().sort_index().to_string())


hard match (exact∪digits): 6/300 = 2.0%
FACE hard-hit rows: 6


soft unmatched capped alloc {2017: 27, 2018: 80, 2019: 15, 2020: 28}
unmatched soft-link candidates U=150 (of 294 unmatched in join set)
unmatched year counts:
 year
2017    27
2018    80
2019    15
2020    28


## 5. Normalização de foro/comarca + pull FACE por mês (baixo custo)

$\phi$: upper + strip acentos + remove `FORO DE/DAS` / `COMARCA DE` + mapear Capital estadual ↔ `Foro das Execuções Fiscais Estaduais`.
Pull FACE em lotes mensais com `distribuicao_data BETWEEN` (±30d) e filtro `foro` por tokens dos places unmatched — evita scan completo por linha.


In [4]:
def norm_place(s) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    t = str(s).upper().strip()
    t = unicodedata.normalize("NFKD", t)
    t = "".join(c for c in t if not unicodedata.combining(c))
    t = re.sub(r"\bFORO\s+(?:DE|DAS|DA|DO|DOS)\s+", " ", t)
    t = re.sub(r"\bCOMARCA\s+(?:DE|DAS|DA|DO|DOS)\s+", " ", t)
    t = re.sub(
        r"^VARA\s+DAS?\s+EXECUCOES\s+FISCAIS\s+ESTADUAIS\b.*",
        "CAPITAL_ESTADUAL",
        t,
    )
    t = re.sub(
        r"^VARA\s+DAS?\s+EXECUCOES\s+FISCAIS\s+MUNICIPAIS\b.*",
        "CAPITAL_MUNICIPAL",
        t,
    )
    t = re.sub(r"\bDA\s+COMARCA\s+(?:DE|DA|DO|DOS)?\s*", " ", t)
    t = re.sub(r"[^A-Z0-9\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    if t in {"EXECUCOES FISCAIS ESTADUAIS", "CAPITAL ESTADUAL"} or (
        "EXECUCOES FISCAIS ESTADUAIS" in t and "MUNICIPAIS" not in t
    ):
        return "CAPITAL_ESTADUAL"
    if t in {"EXECUCOES FISCAIS MUNICIPAIS", "CAPITAL MUNICIPAL"} or "EXECUCOES FISCAIS MUNICIPAIS" in t:
        return "CAPITAL_MUNICIPAL"
    if t in {"CAPITAL", "SAO PAULO"} or t.endswith(" CAPITAL"):
        return "CAPITAL_ESTADUAL"
    return t


def place_search_tokens(place: str) -> list[str]:
    # Tokens / substrings for DuckDB ILIKE foro filter (no PII).
    if not place:
        return []
    if place == "CAPITAL_ESTADUAL":
        return ["Execuções Fiscais Estaduais", "Execucoes Fiscais Estaduais"]
    if place == "CAPITAL_MUNICIPAL":
        return ["Execuções Fiscais Municipais", "Execucoes Fiscais Municipais"]
    parts = [p for p in place.split() if p not in {"DE", "DA", "DO", "DAS", "DOS", "E"}]
    if not parts:
        return [place]
    if len(parts) == 1:
        return [parts[0].title()]
    return [place.title(), " ".join(parts[-2:]).title(), parts[-1].title()]


unm = unm.copy()
unm["place"] = unm["NOMECOMARCA"].map(norm_place)
print("unmatched place distribution (counts only):")
print(unm["place"].value_counts().head(15).to_string())
n_empty_place = int((unm["place"] == "").sum())
print("empty place after norm:", n_empty_place)

unm["ym"] = unm["DT_AJUIZAMENTO"].dt.to_period("M")
face_chunks: list[pd.DataFrame] = []
pull_stats = []

for ym, g in unm.groupby("ym"):
    dt0 = (ym.to_timestamp() - pd.Timedelta(days=MAX_N)).normalize()
    dt1 = (ym.to_timestamp() + pd.offsets.MonthEnd(0) + pd.Timedelta(days=MAX_N)).normalize()
    tokens: set[str] = set()
    for pl in g["place"].unique():
        for tok in place_search_tokens(pl):
            if tok and len(tok) >= 3:
                tokens.add(tok)
    if not tokens:
        pull_stats.append({"ym": str(ym), "n_pef": len(g), "n_face": 0, "tokens": 0})
        continue
    ors = " OR ".join(
        f"lower(CAST(foro AS VARCHAR)) LIKE '%{tok.lower().replace(chr(39), chr(39)+chr(39))}%'"
        for tok in sorted(tokens)
    )
    q = f'''
    SELECT DISTINCT
      CAST(numero AS VARCHAR) AS numero,
      CAST(num_processo_limpo AS VARCHAR) AS num_processo_limpo,
      CAST(foro AS VARCHAR) AS foro,
      CAST(distribuicao_data AS TIMESTAMP) AS distribuicao_data,
      valor_limpo
    FROM delta_scan('{face_path}')
    WHERE distribuicao_data BETWEEN TIMESTAMP '{dt0.date()}' AND TIMESTAMP '{dt1.date()}'
      AND foro IS NOT NULL
      AND ({ors})
    '''
    chunk = con.execute(q).fetchdf()
    chunk["distribuicao_data"] = pd.to_datetime(chunk["distribuicao_data"], errors="coerce")
    chunk["place"] = chunk["foro"].map(norm_place)
    face_chunks.append(chunk)
    pull_stats.append({"ym": str(ym), "n_pef": len(g), "n_face": len(chunk), "tokens": len(tokens)})

pull_df = pd.DataFrame(pull_stats)
print("monthly FACE pulls:")
display(pull_df)
print("total FACE rows pulled (sum months, pre-dedup):", int(pull_df["n_face"].sum()))

if face_chunks:
    face_soft = pd.concat(face_chunks, ignore_index=True)
    face_soft = face_soft.drop_duplicates(subset=["numero", "distribuicao_data", "foro"], keep="first")
else:
    face_soft = pd.DataFrame(
        columns=["numero", "num_processo_limpo", "foro", "distribuicao_data", "valor_limpo", "place"]
    )
print("FACE soft pool unique rows:", len(face_soft))
print("FACE soft pool places (top):")
print(face_soft["place"].value_counts().head(12).to_string() if len(face_soft) else "(empty)")


unmatched place distribution (counts only):
place
CAPITAL_ESTADUAL         111
BARUERI                    7
GUARULHOS                  5
TAUBATE                    4
SAO BERNARDO DO CAMPO      3
CAMPINAS                   3
RIBEIRAO PIRES             2
MAUA                       2
ARARAS                     1
HORTOLANDIA                1
SANTO ANDRE                1
BEBEDOURO                  1
SUZANO                     1
ITAQUAQUECETUBA            1
MAIRIPORA                  1
empty place after norm: 0


monthly FACE pulls:


,ym,n_pef,n_face,tokens
0,2017-03,17,16834,14
1,2017-04,6,6899,7
2,2017-10,1,0,1
3,2017-11,3,6677,2
4,2018-11,69,529,2
5,2018-12,11,518,2
6,2019-03,5,13,2
7,2019-07,1,656,2
8,2019-08,9,699,4
9,2020-08,1,7,3


total FACE rows pulled (sum months, pre-dedup): 38130
FACE soft pool unique rows: 30414
FACE soft pool places (top):


place
CAPITAL_ESTADUAL         8062
GUARULHOS                7448
SAO BERNARDO DO CAMPO    5494
SUZANO                   4352
BEBEDOURO                1201
BARUERI                   784
CAMPO LIMPO PAULISTA      767
SAO JOSE DO RIO PRETO     645
CAMPINAS                  640
RIBEIRAO PIRES            603
SAO JOSE DOS CAMPOS       171
GUARAREMA                 108


## 6. Soft-link scoring por $N$ — unique / ambiguous / none

Para cada $N$, junta unmatched×FACE no mesmo $\phi(\cdot)$ e $|\Delta t|\le N$. Conta $|B_N(i)|$. Sem PII.


In [5]:
def score_blocks(unmatched: pd.DataFrame, face: pd.DataFrame, n_days: int) -> pd.DataFrame:
    out_rows = []
    face_by_place = {pl: g for pl, g in face.groupby("place")} if len(face) else {}
    for r in unmatched.itertuples():
        pl = r.place
        dt = r.DT_AJUIZAMENTO
        cand = face_by_place.get(pl)
        if cand is None or len(cand) == 0 or pd.isna(dt) or not pl:
            if pl and len(face):
                mask = face["place"].map(
                    lambda fp, _pl=pl: bool(fp) and (_pl == fp or _pl in fp or fp in _pl)
                )
                cand = face.loc[mask]
            else:
                cand = face.iloc[0:0]
        if len(cand) == 0:
            out_rows.append({
                "pef": r.PEF, "place": pl, "n_days": n_days,
                "block_size": 0, "status": "none",
            })
            continue
        delta = (cand["distribuicao_data"] - dt).abs().dt.days
        hit = cand.loc[delta <= n_days]
        bs = int(len(hit))
        if bs == 0:
            status = "none"
        elif bs == 1:
            status = "unique"
        else:
            status = "ambiguous"
        out_rows.append({
            "pef": r.PEF, "place": pl, "n_days": n_days,
            "block_size": bs, "status": status,
            "delta_min_days": float(delta.min()) if len(cand) else np.nan,
        })
    return pd.DataFrame(out_rows)


score_parts = [score_blocks(unm, face_soft, n) for n in N_DAYS]
scores = pd.concat(score_parts, ignore_index=True)

summary_N = (
    scores.groupby("n_days")
    .agg(
        U=("pef", "size"),
        n_unique=("status", lambda s: int((s == "unique").sum())),
        n_ambiguous=("status", lambda s: int((s == "ambiguous").sum())),
        n_none=("status", lambda s: int((s == "none").sum())),
        block_size_p50=("block_size", "median"),
        block_size_p90=("block_size", lambda s: float(np.nanpercentile(s, 90))),
        block_size_max=("block_size", "max"),
    )
    .reset_index()
)
summary_N["rate_unique"] = summary_N["n_unique"] / summary_N["U"]
summary_N["rate_ambiguous"] = summary_N["n_ambiguous"] / summary_N["U"]
summary_N["rate_none"] = summary_N["n_none"] / summary_N["U"]
summary_N["cov_hard_only"] = rate_hard
summary_N["cov_hard_plus_unique"] = (n_hard + summary_N["n_unique"]) / n_pef
summary_N["lift_vs_hard"] = summary_N["cov_hard_plus_unique"] / rate_hard

print("=== SOFT-LINK RATES BY N ===")
display(summary_N)

ex7 = scores.loc[scores["n_days"] == 7, "block_size"].value_counts().sort_index().head(20)
print("block_size histogram (N=7), counts only:")
print(ex7.to_string())

ct7 = (
    scores.loc[scores["n_days"] == 7]
    .groupby(["place", "status"], dropna=False)
    .size()
    .rename("n")
    .reset_index()
    .sort_values("n", ascending=False)
)
print("place × status (N=7), top 20 — counts only:")
display(ct7.head(20))


=== SOFT-LINK RATES BY N ===


,n_days,U,n_unique,n_ambiguous,n_none,block_size_p50,block_size_p90,block_size_max,rate_unique,rate_ambiguous,rate_none,cov_hard_only,cov_hard_plus_unique,lift_vs_hard
0,0,150,7,115,28,38.0,114.0,818,0.046667,0.766667,0.186667,0.02,0.043333,2.166667
1,3,150,9,125,16,132.5,317.0,3132,0.060000,0.833333,0.106667,0.02,0.050000,2.500000
2,7,150,2,134,14,181.0,474.0,4786,0.013333,0.893333,0.093333,0.02,0.026667,1.333333
3,15,150,1,136,13,498.0,873.8,5150,0.006667,0.906667,0.086667,0.02,0.023333,1.166667
4,30,150,0,138,12,515.0,1238.0,5200,0.000000,0.920000,0.080000,0.02,0.020000,1.000000


block_size histogram (N=7), counts only:
block_size
0      14
1       2
2       4
3       1
4       1
6       1
9       4
14      1
21      1
26      2
34      4
70      2
95      1
158    35
179     1
181     3
196     4
197     1
200     1
206     2
place × status (N=7), top 20 — counts only:


,place,status,n
5,CAPITAL_ESTADUAL,ambiguous,111
2,BARUERI,ambiguous,7
8,GUARULHOS,ambiguous,5
20,TAUBATE,none,4
4,CAMPINAS,ambiguous,3
16,SAO BERNARDO DO CAMPO,ambiguous,3
13,RIBEIRAO PIRES,ambiguous,2
12,MAUA,none,2
19,SUZANO,none,1
18,SAO JOSE DOS CAMPOS,unique,1


## 7. Figuras Plotly — interpretação

- **Fig. 1 (barras empilhadas por $N$):** composição unique / ambiguous / none. Se ambiguous cresce mais rápido que unique com $N$, a janela alarga o bloco sem resolver identidade.
- **Fig. 2 (linhas):** $\hat{u}_N$ e $\hat{a}_N$ vs $N$. Útil para escolher $N$ operacional (trade-off recall×precisão).
- **Fig. 3 (lift):** cobertura hard-only vs hard+unique soft. Lift alto com $\hat{a}_N$ alto = enriquecimento ruidoso.
- **Fig. 4 (histograma $|B_7|$):** distribuição de tamanhos de bloco em $N=7$; cauda longa ⇒ many-to-one.
- **Fig. 5 (donut hard vs soft outcomes em $N=7$):** visão do join set completo (hard matched + soft unique/ambiguous/none).


In [6]:
melt = summary_N.melt(
    id_vars=["n_days"],
    value_vars=["rate_unique", "rate_ambiguous", "rate_none"],
    var_name="outcome",
    value_name="rate",
)
melt["outcome"] = melt["outcome"].str.replace("rate_", "", regex=False)
fig1 = px.bar(
    melt, x="n_days", y="rate", color="outcome",
    title="Fig. 1 — Soft-link outcomes por janela N (só unmatched)",
    labels={"n_days": "N (dias)", "rate": "taxa", "outcome": "status"},
    barmode="stack",
)
fig1.update_layout(yaxis_tickformat=".0%")
show_and_save(fig1, "softlink_fig01_outcomes_by_N", height=420)

fig2 = go.Figure()
fig2.add_scatter(name="unique û_N", x=summary_N["n_days"], y=summary_N["rate_unique"], mode="lines+markers")
fig2.add_scatter(name="ambiguous â_N", x=summary_N["n_days"], y=summary_N["rate_ambiguous"], mode="lines+markers")
fig2.update_layout(
    title="Fig. 2 — Taxas unique vs ambiguous vs N",
    xaxis_title="N (dias)", yaxis_title="taxa", yaxis_tickformat=".0%",
)
show_and_save(fig2, "softlink_fig02_unique_vs_ambiguous", height=420)

fig3 = go.Figure()
fig3.add_bar(name="hard-only", x=summary_N["n_days"].astype(str), y=summary_N["cov_hard_only"])
fig3.add_bar(name="hard+unique soft", x=summary_N["n_days"].astype(str), y=summary_N["cov_hard_plus_unique"])
fig3.update_layout(
    barmode="group",
    title="Fig. 3 — Cobertura descritiva: hard-only vs hard+unique soft",
    xaxis_title="N (dias)", yaxis_title="cobertura / join set", yaxis_tickformat=".0%",
)
show_and_save(fig3, "softlink_fig03_lift_coverage", height=420)

bs7 = scores.loc[scores["n_days"] == 7, "block_size"]
bs7_plot = bs7.clip(upper=50)
fig4 = px.histogram(
    pd.DataFrame({"block_size": bs7_plot}),
    x="block_size", nbins=26,
    title="Fig. 4 — Distribuição |B_7| (block size; valores >50 truncados na figura)",
    labels={"block_size": "|B_7|", "count": "n PEFs unmatched"},
)
show_and_save(fig4, "softlink_fig04_blocksize_N7", height=420)
print("Fig.4 raw block_size describe:\n", bs7.describe().to_string())

row7 = summary_N.loc[summary_N["n_days"] == 7].iloc[0]
n_unm_total = int((~aj_cov["hard_matched"]).sum())
n_out_cap = n_unm_total - U
pie_df = pd.DataFrame({
    "status": ["hard_matched", "soft_unique", "soft_ambiguous", "soft_none"],
    "n": [
        n_hard,
        int(row7["n_unique"]),
        int(row7["n_ambiguous"]),
        int(row7["n_none"]) + n_out_cap,
    ],
})
fig5 = px.pie(
    pie_df, names="status", values="n", hole=0.45,
    title=f"Fig. 5 — Join set (N=7): hard={n_hard}/{n_pef}; soft sobre U={U} (out-of-cap unmatched→none)",
)
show_and_save(fig5, "softlink_fig05_joinset_donut_N7", height=440)
display(pie_df)


static → /Users/etorebraga/Code/cemepi-ctf-intel-fiscal/.local/figures/litigancia/softlink_fig01_outcomes_by_N.png


static → /Users/etorebraga/Code/cemepi-ctf-intel-fiscal/.local/figures/litigancia/softlink_fig02_unique_vs_ambiguous.png


static → /Users/etorebraga/Code/cemepi-ctf-intel-fiscal/.local/figures/litigancia/softlink_fig03_lift_coverage.png


static → /Users/etorebraga/Code/cemepi-ctf-intel-fiscal/.local/figures/litigancia/softlink_fig04_blocksize_N7.png
Fig.4 raw block_size describe:
 count     150.000000
mean      366.860000
std       653.101892
min         0.000000
25%       110.750000
50%       181.000000
75%       470.000000
max      4786.000000


static → /Users/etorebraga/Code/cemepi-ctf-intel-fiscal/.local/figures/litigancia/softlink_fig05_joinset_donut_N7.png


,status,n
0,hard_matched,6
1,soft_unique,2
2,soft_ambiguous,134
3,soft_none,158


## 8. Conclusão honesta

Interpretamos $\hat{u}_N$ e $\hat{a}_N$ juntos: soft-link **útil para enriquecimento de paper** só se unique for material **e** ambiguous não dominar (especialmente Capital / foros densos). Caso contrário, é diagnóstico de esparsidade do scrape + homonímia temporal — demasiado ruidoso para tratar como join de produção.


In [7]:
best = summary_N.sort_values(["rate_unique", "rate_ambiguous"], ascending=[False, True]).iloc[0]
print("=== CONCLUSION SNAPSHOT ===")
print({
    "n_join": n_pef,
    "n_hard": n_hard,
    "rate_hard": round(rate_hard, 4),
    "U_soft": U,
    "best_N": int(best["n_days"]),
    "best_unique_rate": round(float(best["rate_unique"]), 4),
    "best_ambiguous_rate": round(float(best["rate_ambiguous"]), 4),
    "best_lift": round(float(best["lift_vs_hard"]), 3),
    "face_soft_pool": len(face_soft),
})

u = float(best["rate_unique"])
a = float(best["rate_ambiguous"])
if u >= 0.10 and u >= a:
    verdict = "usable_with_caution"
    note = (
        "Unique soft-links ≥10% e ≥ ambiguous no melhor N: candidato a enriquecimento "
        "exploratório de paper, com revisão manual / validação cruzada."
    )
elif u >= 0.05 and a < 0.5:
    verdict = "marginal"
    note = (
        "Lift único modesto; ambiguidade ainda relevante. Útil como diagnóstico, "
        "não como join de produção."
    )
else:
    verdict = "too_noisy"
    note = (
        "Ambiguous/none dominam: foro+data sem chave CNJ é demasiado ruidoso "
        "(homonímia temporal, Capital densa, scrape incompleto)."
    )
print("verdict:", verdict)
print("note:", note)

report = {
    "extracao": f"{S.ano:04d}-{S.mes:02d}",
    "window": f"{WINDOW_START.date()}…{WINDOW_END.date()}",
    "n_join": int(n_pef),
    "n_hard": int(n_hard),
    "rate_hard": float(rate_hard),
    "U_soft": int(U),
    "summary_by_N": summary_N.to_dict(orient="records"),
    "verdict": verdict,
    "note": note,
    "fig_dir": str(FIG_DIR),
}
report_path = SAMPLE_DIR / "toy_softlink_summary.json"
try:
    report_path.write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
    print("wrote", report_path)
except Exception as e:  # noqa: BLE001
    print("skip json write", e)

rates_csv = FIG_DIR / "softlink_rates_by_N.csv"
summary_N.to_csv(rates_csv, index=False)
print("rates csv", rates_csv)


=== CONCLUSION SNAPSHOT ===
{'n_join': 300, 'n_hard': 6, 'rate_hard': 0.02, 'U_soft': 150, 'best_N': 3, 'best_unique_rate': 0.06, 'best_ambiguous_rate': 0.8333, 'best_lift': 2.5, 'face_soft_pool': 30414}
verdict: too_noisy
note: Ambiguous/none dominam: foro+data sem chave CNJ é demasiado ruidoso (homonímia temporal, Capital densa, scrape incompleto).
wrote /Volumes/Meedi_Etore_HD1/CEMEPI/dumps/cemepi_api/extracao=2026-03/samples/toy_softlink_summary.json
rates csv /Users/etorebraga/Code/cemepi-ctf-intel-fiscal/.local/figures/litigancia/softlink_rates_by_N.csv


## 9. Limitações

- Soft-link **não** prova identidade processual; só bloqueio geotemporal.
- Capital / foros densos geram $|B_N|\gg 1$ mesmo com $N$ pequeno.
- Dump `NOMECOMARCA` ≠ FACE `foro` lexicalmente (Vara vs Foro; estadual vs municipal).
- Amostra toy ≠ cobertura estadual; DuckDB pulls mensais ainda dependem de tokens de foro.
- Sem EM / Fellegi–Sunter weights; `valor_limpo` puxado mas não usado no score 1:1 deste toy.
- Sem PII; sem commit.


## 10. Referências (framing)

- Fellegi, I. P., & Sunter, A. B. (1969). A theory for record linkage. *JASA*, 64(328), 1183–1210. https://doi.org/10.1080/01621459.1969.10501049 — framing exact vs probabilistic; **aqui:** blocking determinístico.
- Christen, P. (2012). *Data Matching*. Springer. https://doi.org/10.1007/978-3-642-31164-2 — normalização e blocking antes do link.


---
**Atribuição.** Conteúdo, experimentos e conclusões: do autor. Assistência de formatação/estruturação: IA.
